## Building a Whole Genome Sequencing (WGS) Assembly AI Agent

In this notebook, I construct a bioinformatics AI agent tool using OpenAI's function calling API. I'll do some research on free alternatives later.

# Workflow Overview:

Tool Setup: Subprocess wrapper for the SPAdes assembler & Python-native N50 contig analyzer.

Schema Definition: JSON definitions informing the LLM about available tools and required parameters.

Agent Core: Orchestration loop processing LLM messages, executing tool requests, and passing output back.

Assembly: Putting everything together to run in one cell.

## 1. Setup and Client
First, dependencies, modules, and LLM Client.

In [ ]:
# Setup and imports
import json
import os
import subprocess
from openai import OpenAI #(We'll make a LLM agnostic module later.)

# Initialize OpenAI client (Ensure LLM_API_KEY environment variable is set)
client = OpenAI(api_key=os.environ.get("LLM_API_KEY"))

## 2. Spades Assembly tool
This tool executes the spades.py CLI command on paired-end FASTQ reads (R1 and R2). It uses Python's subprocess module to manage execution safely and capture stdout/stderr

In [5]:
def run_spades_assembly(r1_path: str, r2_path: str, output_dir: str, threads: int = 4) -> dict:
    """Runs SPAdes de novo genome assembler on paired-end reads."""
    if not os.path.exists(r1_path) or not os.path.exists(r2_path):
        return {"error": f"Input FASTQ files missing: {r1_path} or {r2_path}"}

    cmd = [
        "spades.py",
        "-1", r1_path,
        "-2", r2_path,
        "-o", output_dir,
        "-t", str(threads)
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        contigs_file = os.path.join(output_dir, "contigs.fasta")

        return {
            "status": "success",
            "output_dir": output_dir,
            "contigs_file": contigs_file,
            "stdout_tail": result.stdout[-500:]
        }
    except subprocess.CalledProcessError as e:
        return {"status": "error", "message": e.stderr[-500:]}
    except FileNotFoundError:
        return {"status": "error", "message": "'spades.py' standard binary was not found in system PATH."}

# 3. Tool 2 — Assembly Metrics & N50 Calculation
After assembly, the agent needs a way to evaluate contig quality. The function below parses the contigs FASTA file and computes total contig count, total base pairs, largest contig size, and N50 length. 

In [6]:
def calculate_assembly_metrics(fasta_path: str) -> dict:
    """Calculates assembly statistics (N50, total bp, max contig size) from FASTA output."""
    if not os.path.exists(fasta_path):
        return {"error": f"FASTA file not found at path: {fasta_path}"}

    lengths = []
    current_len = 0
    with open(fasta_path, "r") as f:
        for line in f:
            if line.startswith(">"):
                if current_len > 0:
                    lengths.append(current_len)
                    current_len = 0
            else:
                current_len += len(line.strip())
        if current_len > 0:
            lengths.append(current_len)

    if not lengths:
        return {"error": "FASTA file is empty or formatted incorrectly."}

    lengths.sort(reverse=True)
    total_length = sum(lengths)
    half_length = total_length / 2.0

    cumulative_len = 0
    n50 = 0
    for length in lengths:
        cumulative_len += length
        if cumulative_len >= half_length:
            n50 = length
            break

    return {
        "total_contigs": len(lengths),
        "total_bp": total_length,
        "max_contig_bp": lengths[0],
        "n50_bp": n50
    }

# 4.Tool Schemas & Tool Dispatch Mapping

For the LLM to know when and how to invoke functions, define JSON schemas specifying each tool's name, description, and required parameter signatures. Map tool names directly to local Python functions in a lookup table.

In [7]:
# Define JSON Schema provided to OpenAI chat model
tools = [
    {
        "type": "function",
        "function": {
            "name": "run_spades_assembly",
            "description": "Executes SPAdes de novo assembly on paired-end FASTQ sequence files.",
            "parameters": {
                "type": "object",
                "properties": {
                    "r1_path": {"type": "string", "description": "Path to Read 1 FASTQ file."},
                    "r2_path": {"type": "string", "description": "Path to Read 2 FASTQ file."},
                    "output_dir": {"type": "string", "description": "Directory path for output assembly."},
                    "threads": {"type": "integer", "description": "Number of CPU threads to allocate."}
                },
                "required": ["r1_path", "r2_path", "output_dir"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_assembly_metrics",
            "description": "Evaluates quality metrics (N50, contig counts, total length) from output FASTA contigs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "fasta_path": {"type": "string", "description": "Path to contigs FASTA file."}
                },
                "required": ["fasta_path"]
            }
        }
    }
]

# Dispatcher dictionary mapping tool strings to callable local functions
TOOL_MAPPING = {
    "run_spades_assembly": run_spades_assembly,
    "calculate_assembly_metrics": calculate_assembly_metrics
}

# 5.Agent Control Loop

The main loop handles multi-turn execution. If the model determines a tool call is needed:

It requests a specific function call with parameters.

The loop runs the Python function locally.

Results are returned to the conversation as a tool role message.

The model continues until no further tools are required, yielding a final response.

In [8]:
def run_wgs_agent(user_prompt: str):
    """Executes stateful agent control loop."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert bioinformatics AI agent. Execute requested WGS assembly tasks "
                "by invoking tools sequentially. Always summarize the final assembly stats."
            )
        },
        {"role": "user", "content": user_prompt}
    ]

    print(f"🤖 User Request: '{user_prompt}'\n")

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        # Check if model requested a tool invocation
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                print(f"🛠️ Agent Executing Action: `{fn_name}` with args: {fn_args}")

                # Execute function locally
                tool_function = TOOL_MAPPING[fn_name]
                tool_result = tool_function(**fn_args)

                # Return tool output back to conversation history
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })
        else:
            # Complete step when no further tools are requested
            print(f"\n📊 Final Output:\n{response_message.content}")
            break

# 6.Complete Assembled Script

The code block below integrates all modular parts into a single self-contained execution script.

In [9]:
import json
import os
import subprocess
from openai import OpenAI

# Initialize client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 1. Tools Implementation
def run_spades_assembly(r1_path: str, r2_path: str, output_dir: str, threads: int = 4) -> dict:
    if not os.path.exists(r1_path) or not os.path.exists(r2_path):
        return {"error": f"Input FASTQ files missing: {r1_path} or {r2_path}"}

    cmd = ["spades.py", "-1", r1_path, "-2", r2_path, "-o", output_dir, "-t", str(threads)]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return {
            "status": "success",
            "output_dir": output_dir,
            "contigs_file": os.path.join(output_dir, "contigs.fasta"),
            "stdout_tail": result.stdout[-500:]
        }
    except subprocess.CalledProcessError as e:
        return {"status": "error", "message": e.stderr[-500:]}
    except FileNotFoundError:
        return {"status": "error", "message": "'spades.py' executable not found in PATH."}

def calculate_assembly_metrics(fasta_path: str) -> dict:
    if not os.path.exists(fasta_path):
        return {"error": f"FASTA file not found at {fasta_path}"}

    lengths = []
    current_len = 0
    with open(fasta_path, "r") as f:
        for line in f:
            if line.startswith(">"):
                if current_len > 0:
                    lengths.append(current_len)
                    current_len = 0
            else:
                current_len += len(line.strip())
        if current_len > 0:
            lengths.append(current_len)

    if not lengths:
        return {"error": "No valid sequence data found."}

    lengths.sort(reverse=True)
    total_length = sum(lengths)
    half_length = total_length / 2.0

    cumulative_len, n50 = 0, 0
    for l in lengths:
        cumulative_len += l
        if cumulative_len >= half_length:
            n50 = l
            break

    return {
        "total_contigs": len(lengths),
        "total_bp": total_length,
        "max_contig_bp": lengths[0],
        "n50_bp": n50
    }

# 2. Tool Declarations & Dispatch Mapping
tools = [
    {
        "type": "function",
        "function": {
            "name": "run_spades_assembly",
            "description": "Executes SPAdes de novo assembly on paired-end FASTQ reads.",
            "parameters": {
                "type": "object",
                "properties": {
                    "r1_path": {"type": "string"},
                    "r2_path": {"type": "string"},
                    "output_dir": {"type": "string"},
                    "threads": {"type": "integer"}
                },
                "required": ["r1_path", "r2_path", "output_dir"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_assembly_metrics",
            "description": "Calculates contig assembly metrics (N50, total bp).",
            "parameters": {
                "type": "object",
                "properties": {
                    "fasta_path": {"type": "string"}
                },
                "required": ["fasta_path"]
            }
        }
    }
]

TOOL_MAPPING = {
    "run_spades_assembly": run_spades_assembly,
    "calculate_assembly_metrics": calculate_assembly_metrics
}

# 3. Agent Execution Engine
def run_wgs_agent(user_prompt: str):
    messages = [
        {"role": "system", "content": "You are a bioinformatics AI agent. Execute assembly and calculate stats."},
        {"role": "user", "content": user_prompt}
    ]

    print(f"🤖 User Prompt: '{user_prompt}'\n")

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                print(f"🛠️ Tool Called: `{fn_name}` | Args: {fn_args}")

                tool_result = TOOL_MAPPING[fn_name](**fn_args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })
        else:
            print(f"\n📊 Summary:\n{response_message.content}")
            break

# 4. Main Run Execution
if __name__ == "__main__":
    prompt = (
        "Run assembly on sample_R1.fastq and sample_R2.fastq into directory './wgs_results'. "
        "Calculate quality metrics and return a summary."
    )
    run_wgs_agent(prompt)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.